In [1]:
import os
import pathlib
import uuid

import mlflow
import pandas as pd

In [2]:
year = 2021
month = 2
taxi_type = "green"

In [3]:
input_file = f"https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_{year:04d}-{month:02d}.parquet"

output_folder = pathlib.Path("output") / taxi_type
output_folder.mkdir(parents=True, exist_ok=True)
output_file = output_folder / f"{year:04d}-{month:02d}.parquet"

run_id = os.getenv("RUN_ID", "7e1c7ff453944da98f670500f11d3bc4")
model_id = os.getenv("MODEL_ID", "m-205aad2b19454838af2cfe5644624f7e")

In [4]:
def generate_uuids(n: int):
    return [str(uuid.uuid4()) for _ in range(n)]


def read_dataframe(filename: str):
    df = pd.read_parquet(filename)

    df["duration"] = df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]
    df["duration"] = df["duration"].dt.total_seconds() / 60
    df = df[(df["duration"] >= 1) & (df["duration"] <= 60)]

    df["ride_id"] = generate_uuids(len(df))

    categorical = ["PULocationID", "DOLocationID"]
    df[categorical] = df[categorical].astype(str)

    df["PU_DO"] = df["PULocationID"] + "_" + df["DOLocationID"]

    return df


def prepare_features(df: pd.DataFrame):
    categorical = ["PU_DO"]
    numerical = ["trip_distance"]
    dicts = df[categorical + numerical].to_dict(orient="records")
    return dicts

In [5]:
def load_model(model_id: str):
    # logged_model = f"runs:/{run_id}/model"
    logged_model = f"models:/{model_id}"
    # logged_model = f"mlflow-artifacts:/3/models/{model_id}/artifacts"
    # logged_model = f"s3://mlflow/3/models/{model_id}/artifacts"
    model = mlflow.pyfunc.load_model(logged_model)
    return model


def predict(model, features):
    return model.predict(features)


def save_results(df, y_pred, model_id, output_file):
    df_result = df[
        ["ride_id", "lpep_pickup_datetime", "PULocationID", "DOLocationID"]
    ].copy()
    df_result["actual_duration"] = df["duration"]
    df_result["predicted_duration"] = y_pred
    df_result["diff"] = df_result["actual_duration"] - df_result["predicted_duration"]
    df_result["model_version"] = model_id

    df_result.to_parquet(output_file, index=False)


def apply_model(input_file: str, model_id: str, output_file: str):
    df = read_dataframe(input_file)
    features = prepare_features(df)

    model = load_model(model_id)
    y_pred = predict(model, features)

    save_results(df, y_pred, model_id, output_file)

In [6]:
apply_model(input_file=input_file, model_id=model_id, output_file=output_file)